<a href="https://colab.research.google.com/github/edwardsnj/pride-study-retrieval-cofest-2026/blob/main/notebooks/TextEmbedding%2BTFIDF_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#
# Import necessary modules
#

import sys, os, httpimport

with httpimport.github_repo("EdwardsLabProjects", "pride-study-retrieval-cofest-2026", ref="main"):
    from notebooks.lib import *

set_random_seed(state=None)

Version: 1.0.25
Using random seed: 3506330


In [2]:
#
# Get embeddings and known true positives
#

md,emb = embeddings("openai-3-small")
embdim = emb.shape[0]
tp,tn = knownstudies()

allacc = set(emb.columns)
tp &= allacc
tn &= allacc

assert len(tp & tn) == 0, "TP and TN should not intersect!"

# Print out some details...
print(f"Embeddings: {emb.shape}")
print(f"True Pos: {len(tp)}")
print(f"True Neg: {len(tn)}")

allacc,avgemb = select_by_embedding_proximity(tp,emb,1000)
print(f"All Acc: {len(allacc)}")

Embeddings: (1536, 36696)
True Pos: 84
True Neg: 47
All Acc: 1000


In [3]:
#
# Create train/test split
#
TESTFRAC = 0.2
BACKGROUND_OVERSAMPLE = 25

train_acc, train_y, test_acc, test_y = split_train_test(allacc, tp, tn,
    test_size=TESTFRAC, bgsize=BACKGROUND_OVERSAMPLE)


In [4]:
# Build the TF-IDF features from train data only,
# but build feature vectors for all documents

from sklearn.feature_extraction import text
stop_words = list(set(list(text.ENGLISH_STOP_WORDS) + """

""".split()))

tfidf_extra_args = dict(positive_only=False, # Use only positive set for word selection
                        min_df=1, # minimum documents word is in
                        max_df=1.0, # maximum documents word is in
                        max_features=None, # max number of words to use
                        stop_words=stop_words) # words to ignore

tfidf,tfidf_model = create_tfidf_features(md, train_acc, train_y,
                                          test_acc, **tfidf_extra_args)

print(f"TF-IDF features shape: {tfidf.shape}")

TF-IDF features shape: (21494, 1000)


In [5]:
logreg_extra_args = dict(class_weight='balanced',C=10.0,
                         penalty='l1', solver='liblinear',
                         use_embed=True, use_tfidf=True)
trained_model = train_document_classifier(emb, tfidf,
                                          train_acc, train_y,
                                          test_acc, test_y,
                                          **logreg_extra_args)
print("NZ Coeffs:",sum(*[1*(xi!=0) for xi in trained_model.coef_]))

Training data shape: (800, 23030) (Positives: 67, Negatives: 733)
Testing data shape: (200, 23030) (Positives: 17, Negatives: 183)

--- Model Evaluation ---
Accuracy: 0.9550

Classification Report:
                precision    recall  f1-score   support

Background (0)       0.97      0.98      0.98       183
 Seed-like (1)       0.75      0.71      0.73        17

      accuracy                           0.95       200
     macro avg       0.86      0.84      0.85       200
  weighted avg       0.95      0.95      0.95       200

NZ Coeffs: 83


In [6]:
nzemb,nztfidf,topfeat = top_features(trained_model,tfidf_model,nembed=embdim,**logreg_extra_args)
print(f"Number of sem. embedding dimensions with non-zero coefficients: {nzemb}")
print(f"Number of TF-IDF dimensions with non-zero coefficients: {nztfidf}")
print("\nTop 20 TF-IDF Features:")
display(topfeat.head(20))

show_top_feature_examples(topfeat, md)


Number of sem. embedding dimensions with non-zero coefficients: 7
Number of TF-IDF dimensions with non-zero coefficients: 76

Top 20 TF-IDF Features:


,Feature,Coefficient
889,18o,32.865467
15730,pngase,24.793111
10553,hydrazide,24.508156
9555,glycosylated,24.111205
8564,fcgriia,21.564139
9518,glycoproteins,21.476678
6668,deamidation,20.351003
13916,neb,18.713471
6155,contortus,-16.417123
19609,thermophilic,15.997689



=== '18o' (coef: +32.8655) ===
  PXD010911: Human serum was obtained from Research Blood Components (RBC), following American Association of Blood Banks guidelines (Boston, MA). Blood from healthy males and females between the ages 18 and 65 was centrifuged to collect plasma and coagulated with ACD. Antibodies from patient serum were enriched using isotype specific CaptureSelect Affinity Matrix resin (ThermoFisher). 10 mL plasma was dialyzed, filtered and added to 500 μL of resuspended and washed resin (based on 2.5-8 mg/mL binding capacity), and incubated for 1 hour with end-to-end mixing, at room temperature. The enriched antibody was eluted using 0.1 M Acetic acid and immediately neutralized with 1 M Tris, pH 7.5. The eluted fraction was concentrated and buffer exchanged into PBS using an Amicon concentrator tube (50-kDa cutoff). Enrichment and purity were confirmed by Luminex analysis (MilliporeSigma) and the concentration of each sample was assessed by human anti-Ig ELISAs (Therm